# MMDP-OD-RTDP Master Report

This notebook serves as the master empirical report for comparing **Baseline Real-Time Dynamic Programming (RTDP)** against **Operator Decomposition (OD) RTDP** in stochastic Multi-Agent Pathfinding environments (MMDP).

We will progress through increasingly difficult maps. For each map, we will benchmark both planners, and then launch an interactive visualization to watch the OD planner execute alongside its branching tree.

### 1. Framework Setup
First, we initialize the highly optimized batch framework and define our benchmarking function.

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

REPO_URL = "https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git"
REPO_NAME = "MMDP-OD-RTDP-PROJECT"

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    sys.path.append('src')
else:
    src_path = str(Path('.').resolve() / 'src')
    if src_path not in sys.path:
        sys.path.append(src_path)
        
from grid_mmdp import GridMMDP, MMDPConfig
from map_creator import create_map_instance
from heuristic import ShortestPathHeuristic
from baseline_rtdp import BaselineRTDP, RTDPConfig
from od_rtdp import OperatorDecompositionRTDP
from evaluation import evaluate_policy, EvaluationConfig
from resource_monitor import ResourceMonitor
from visualizer import TrajectoryVisualizer

# Global array to aggregate results across all notebook cells
GLOBAL_RESULTS = []

def run_comparison(map_path: str, n_agents: int, time_limit: float = 5.0, episodes: int = 50):
    map_instance = create_map_instance(map_path, scenario_number=1, task_offset=0, n_agents=n_agents)
    mdp = GridMMDP(map_instance, MMDPConfig(slip_to_stay_probability=0.20))
    heuristic = ShortestPathHeuristic(mdp)
    
    print(f"\n{'='*60}\nRunning Benchmark: {map_instance.grid_map.name} ({n_agents} Agents) | Time Limit: {time_limit}s\n{'='*60}")
    
    config = RTDPConfig(time_limit_seconds=time_limit, seed=42)
    eval_config = EvaluationConfig(episodes=episodes, seed=101)
    
    # Baseline RTDP
    baseline_planner = BaselineRTDP(mdp, heuristic, config)
    print("Running Baseline RTDP...")
    with ResourceMonitor() as monitor:
        baseline_result = baseline_planner.solve()
    baseline_mem = monitor.snapshot().peak_rss_delta_mb
    baseline_eval = evaluate_policy(mdp, baseline_planner, eval_config)
    
    # OD RTDP
    od_planner = OperatorDecompositionRTDP(mdp, heuristic, config)
    print("Running OD RTDP...")
    with ResourceMonitor() as monitor:
        od_result = od_planner.solve()
    od_mem = monitor.snapshot().peak_rss_delta_mb
    od_eval = evaluate_policy(mdp, od_planner, eval_config)
    
    # Print Mini Summary
    print(f"\n[BASELINE] Success: {baseline_eval.summary.success_rate*100:.1f}% | Backups: {baseline_result.bellman_backups:,} | RAM: {baseline_mem:.1f} MB")
    print(f"[OD RTDP]  Success: {od_eval.summary.success_rate*100:.1f}% | Backups: {od_result.bellman_backups:,} | RAM: {od_mem:.1f} MB")
    
    metrics = {
        "map": map_instance.grid_map.name,
        "agents": n_agents,
        "baseline": {
            "trials": baseline_result.trials_completed,
            "backups": baseline_result.bellman_backups,
            "success": baseline_eval.summary.success_rate,
            "steps": baseline_eval.summary.mean_steps_successful_episodes,
            "memory_mb": baseline_mem
        },
        "od": {
            "trials": od_result.trials_completed,
            "backups": od_result.bellman_backups,
            "success": od_eval.summary.success_rate,
            "steps": od_eval.summary.mean_steps_successful_episodes,
            "memory_mb": od_mem
        }
    }
    
    return metrics, mdp, od_planner

print("✅ Framework loaded successfully.")

## Experiment 1: The Curse of Dimensionality (Agent Scaling)
We test an open 8x8 grid scaling from 2 up to 10 agents. As we add agents, Baseline RTDP's branching factor $|A|^n$ explodes (25 $\to$ 125 $\to$ 625), whereas OD RTDP's branching factor $|A| \times n$ grows linearly (10 $\to$ 15 $\to$ 20).

In [ ]:
for agents in range(2, 11):
    metrics, mdp, planner = run_comparison('maps/empty-8-8', n_agents=agents, time_limit=5.0)
    GLOBAL_RESULTS.append(metrics)

print("\n--- Visualizing OD Execution (4 Agents) ---")
viz_1 = TrajectoryVisualizer(mdp, planner)
viz_1.show_with_tree()

## Experiment 2: Diagnostic Crossing 9x9 (2-4 Agents)
This map forces the agents to cross through a central intersection, creating artificial conflict bottlenecks.

In [ ]:
for agents in range(2, 5):
    metrics, mdp, planner = run_comparison('maps/diagnostic/crossing-9-9', n_agents=agents, time_limit=5.0)
    GLOBAL_RESULTS.append(metrics)

print("\n--- Visualizing OD Execution ---")
viz_2 = TrajectoryVisualizer(mdp, planner)
viz_2.show_with_tree()

## Experiment 3: Complex Warehouse (2-10 Agents)
This is a massive warehouse structure with long corridors. We allow 10 seconds of planning time to handle the vast state space.

In [ ]:
for agents in range(2, 11):
    metrics, mdp, planner = run_comparison('maps/warehouse-10-20-10-2-1', n_agents=agents, time_limit=10.0)
    GLOBAL_RESULTS.append(metrics)

print("\n--- Visualizing OD Execution ---")
viz_3 = TrajectoryVisualizer(mdp, planner)
viz_3.show_with_tree()

## Final Aggregation & Analysis
With all map problems completed, we now aggregate the metrics collected in the global array to visualize the massive performance differences between Baseline and Operator Decomposition RTDP.

In [ ]:
import pandas as pd

rows = []
for res in GLOBAL_RESULTS:
    # Baseline Row
    rows.append({
        'Map': res['map'],
        'Agents': res['agents'],
        'Algorithm': 'Baseline RTDP',
        'Success Rate (%)': res['baseline']['success'] * 100,
        'Bellman Backups': res['baseline']['backups'],
        'Peak RAM (MB)': res['baseline']['memory_mb']
    })
    # OD Row
    rows.append({
        'Map': res['map'],
        'Agents': res['agents'],
        'Algorithm': 'OD RTDP',
        'Success Rate (%)': res['od']['success'] * 100,
        'Bellman Backups': res['od']['backups'],
        'Peak RAM (MB)': res['od']['memory_mb']
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Filter empty-8-8 for scaling lineplots
df_scale = df[df['Map'] == 'empty-8-8'].copy()

# 1. Bellman Backups (Log Scale) Lineplot
sns.lineplot(data=df_scale, x='Agents', y='Bellman Backups', hue='Algorithm', 
             marker='o', linewidth=2, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Computation Scaling (empty-8-8)')
axes[0].set_xticks(range(2, 11))

# 2. Memory Scaling Lineplot
sns.lineplot(data=df_scale, x='Agents', y='Peak RAM (MB)', hue='Algorithm', 
             marker='o', linewidth=2, palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Memory Scaling (empty-8-8)')
axes[1].set_xticks(range(2, 11))

# 3. Success Rate Barplot (all maps, 3 agents)
df_3ag = df[df['Agents'] == 3].copy()
sns.barplot(data=df_3ag, x='Map', y='Success Rate (%)', hue='Algorithm', 
            palette=['#e74c3c', '#2ecc71'], ax=axes[2])
axes[2].set_title('Robustness on Different Maps (3 Agents)')
axes[2].set_ylim(0, 105)
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

## Conclusion

### The Curse of Dimensionality
Baseline RTDP struggles because the branching factor of the joint-action space grows exponentially ($|A|^n$). With 3 agents having 5 moves each, the branching factor is 125. This causes extreme memory pressure and slows down state expansion, severely limiting the number of Bellman backups that can be completed.

### The OD Solution
Operator Decomposition mitigates this by breaking simultaneous joint actions into a sequence of individual decisions. The branching factor drops to $|A| \times n = 15$. As demonstrated by the animated trees above, the state-space tree is much narrower, allowing the planner to compute vastly more Bellman backups using a fraction of the RAM. 

As proven in the charts, this structural advantage consistently yields highly robust policies even on complex bottleneck maps like the Warehouse.

## Appendix: Interactive Sandbox
Below is a self-contained interactive dashboard. You can draw your own maps by toggling cells as Obstacles, or placing Agent Starts/Goals. Then click **Run OD-RTDP** to see the actual project planners navigate the map!

In [ ]:
from sandbox import InteractiveSandbox

sandbox = InteractiveSandbox(initial_grid_size=8, max_agents=4)
sandbox.show()